# Complete EnerGIS Workflow - Network Designer to Simulation

## End-to-End Beispiel: Thermisches Netzwerk planen, optimieren, und analysieren

Dieses Notebook zeigt den **kompletten Workflow** von:
1. ✅ **Network Designer**: Komponenten visuell platzieren
2. ✅ **YAML Export**: Konfiguration speichern
3. ✅ **Simulation**: Optimierung mit Pyomo/Gurobi
4. ✅ **Results**: Ergebnisse analysieren und visualisieren

---

## Voraussetzungen

Stellen Sie sicher, dass alles installiert ist:

```bash
# System-Check durchführen
python check_system.py

# Falls Fehler:
pip install -r requirements.txt
pip install -e .
```

---

## Teil 1: Setup & Imports

In [ ]:
# Standard imports
from pathlib import Path
import yaml
import pandas as pd
import matplotlib.pyplot as plt

# EnerGIS Framework
from energis.io.network_designer import create_network_designer
from energis.run.rolling_horizon import run_workflow
from energis.io.dashboard import create_dashboard

print("✅ Imports successful")

---

## Teil 2: Netzwerk im Designer erstellen

### Option A: Programmatisch (in diesem Notebook)

In [ ]:
# Network Designer instanz erstellen
designer = create_network_designer()

print(f"Network Designer erstellt")
print(f"Komponenten: {len(designer.components)}")
print(f"Verbindungen: {len(designer.connections)}")

### Beispiel-Szenario: Brownfield-Ausbau

**Ausgangssituation:**
- Bestandskessel: 20 MW
- Neue Wärmepumpe: zu optimieren
- Neuer Speicher: zu optimieren
- Verbraucher: 25 MW Spitzenlast

In [ ]:
# 1. Bestandskessel (links unten)
designer.add_component(x=100, y=100, comp_type='boiler')
boiler = designer.components[0]
boiler.component_id = 'Kessel_Bestand'
boiler.status = 'existing'
boiler.properties['capacity_mw'] = 20.0
boiler.properties['efficiency'] = 0.95

# 2. Neue Wärmepumpe (links Mitte)
designer.add_component(x=100, y=300, comp_type='heat_pump')
hp = designer.components[1]
hp.component_id = 'WP_Neu'
hp.status = 'investment'  # Wird optimiert!
hp.properties['capacity_mw'] = 15.0  # Startkapazität
hp.properties['cop'] = 3.8

# 3. Neuer Speicher (Mitte)
designer.add_component(x=400, y=200, comp_type='storage')
storage = designer.components[2]
storage.component_id = 'Speicher_Neu'
storage.status = 'investment'  # Wird optimiert!
storage.properties['capacity_mwh'] = 75.0
storage.properties['efficiency'] = 0.98

# 4. Verbraucher (rechts)
designer.add_component(x=700, y=200, comp_type='consumer')
consumer = designer.components[3]
consumer.component_id = 'Waermenetz'
consumer.properties['demand_mw'] = 25.0

print("✅ 4 Komponenten erstellt:")
for comp in designer.components:
    print(f"  - {comp.component_id}: {comp.status} @ ({comp.x}, {comp.y})")

In [ ]:
# Verbindungen erstellen
designer.add_connection(boiler.component_id, storage.component_id)
designer.add_connection(hp.component_id, storage.component_id)
designer.add_connection(storage.component_id, consumer.component_id)

print("✅ 3 Verbindungen erstellt:")
for conn in designer.connections:
    print(f"  {conn.from_id} → {conn.to_id}")

In [ ]:
# Validierung
valid, errors = designer.validate_network()

if valid:
    print("✅ Netzwerk-Validierung erfolgreich!")
else:
    print("❌ Validierungsfehler:")
    for err in errors:
        print(f"  {err}")

### Option B: Interaktives Dashboard (empfohlen für erste Benutzung)

**Terminal:**
```bash
python start_network_designer.py
```

Dann:
1. Komponenten mit Maus platzieren
2. Eigenschaften konfigurieren
3. Exportieren
4. Hier fortfahren mit Teil 3

---

## Teil 3: YAML Export

In [ ]:
# Export-Pfad
export_path = Path('exports/brownfield_example.yaml')
export_path.parent.mkdir(parents=True, exist_ok=True)

# Exportieren
config = designer.export_to_yaml(export_path)

print(f"✅ Exportiert nach: {export_path}")
print(f"\nKonfiguration enthält:")
print(f"  - {len(config.get('system', {}).get('heat_pumps', []))} Wärmepumpen")
print(f"  - {1 if config.get('system', {}).get('storage', {}).get('enabled') else 0} Speicher")
print(f"  - {len(config.get('system', {}).get('generators', {}))} Kessel")
print(f"  - Run mode: {config.get('scenario', {}).get('run_mode')}")

### YAML Vorschau (erste 100 Zeilen)

In [ ]:
# YAML anzeigen
with open(export_path, 'r') as f:
    lines = f.readlines()
    print(''.join(lines[:100]))
    if len(lines) > 100:
        print(f"\n... ({len(lines)-100} weitere Zeilen)")

---

## Teil 4: Zeitreihen-Daten vorbereiten

**WICHTIG:** Für die Simulation benötigen wir Zeitreihen-Daten!

### Option A: Existierende Excel-Daten

In [ ]:
# Prüfen ob Daten verfügbar
data_paths = [
    'data/Import_Data.xlsx',
    'notebooks/data/stadtbach_input.xlsx',
    'examples/data/example_timeseries.xlsx',
]

data_file = None
for path in data_paths:
    if Path(path).exists():
        data_file = path
        print(f"✅ Daten gefunden: {path}")
        break

if not data_file:
    print("⚠️  Keine Zeitreihen-Daten gefunden!")
    print("   Bitte Excel-Datei mit folgenden Spalten bereitstellen:")
    print("   - waermebedarf_MWth")
    print("   - strompreis_EUR_MWh")
    print("   - grid_co2_kg_MWh")
    print("   - WRG1_T_K (Abwärmequelle Temperatur)")
    print("   - WRG1_Q_cap (Abwärmequelle Kapazität)")

### Option B: Synthetische Testdaten generieren

In [ ]:
# Generiere synthetische Daten für Test (1 Woche)
if not data_file:
    import numpy as np

    hours = 168  # 1 Woche
    
    # Lastprofil (Sinus mit Rauschen)
    base_load = 15.0  # MW
    amplitude = 8.0
    demand = base_load + amplitude * (np.sin(np.linspace(0, 7*2*np.pi, hours)) + 0.2 * np.random.randn(hours))
    demand = np.maximum(demand, 5.0)  # Min 5 MW
    
    # Strompreis (Tag/Nacht Muster)
    price_base = 80.0
    price_amplitude = 30.0
    electricity_price = price_base + price_amplitude * np.sin(np.linspace(0, 7*2*np.pi, hours))
    
    # CO2 Emissionen (korreliert mit Preis)
    co2 = 300.0 + 100.0 * np.sin(np.linspace(0, 7*2*np.pi, hours))
    
    # WRG Quelle (konstant)
    wrg_temp = np.full(hours, 308.15)  # 35°C
    wrg_cap = np.full(hours, 50.0)  # 50 MW
    
    # DataFrame erstellen
    df = pd.DataFrame({
        'waermebedarf_MWth': demand,
        'strompreis_EUR_MWh': electricity_price,
        'grid_co2_kg_MWh': co2,
        'WRG1_T_K': wrg_temp,
        'WRG1_Q_cap': wrg_cap,
    })
    
    # Speichern
    data_file = 'exports/synthetic_timeseries.xlsx'
    df.to_excel(data_file, index=False)
    
    print(f"✅ Synthetische Daten generiert: {data_file}")
    print(f"   Dauer: {hours} Stunden (1 Woche)")
    print(f"   Wärmebedarf: {demand.min():.1f} - {demand.max():.1f} MW")
    print(f"   Strompreis: {electricity_price.min():.1f} - {electricity_price.max():.1f} EUR/MWh")

# Daten anzeigen
df_preview = pd.read_excel(data_file, nrows=5)
print("\nDaten-Vorschau:")
display(df_preview)

### Daten visualisieren

In [ ]:
df_full = pd.read_excel(data_file)

fig, axes = plt.subplots(3, 1, figsize=(12, 8))

# Wärmebedarf
axes[0].plot(df_full['waermebedarf_MWth'], 'b-', linewidth=2)
axes[0].set_ylabel('Wärmebedarf (MW)')
axes[0].set_title('Zeitreihen-Daten')
axes[0].grid(True, alpha=0.3)

# Strompreis
axes[1].plot(df_full['strompreis_EUR_MWh'], 'g-', linewidth=2)
axes[1].set_ylabel('Strompreis (EUR/MWh)')
axes[1].grid(True, alpha=0.3)

# CO2
axes[2].plot(df_full['grid_co2_kg_MWh'], 'r-', linewidth=2)
axes[2].set_ylabel('Grid CO2 (kg/MWh)')
axes[2].set_xlabel('Zeitschritt (Stunden)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nStatistik:")
print(f"  Stunden: {len(df_full)}")
print(f"  Gesamt-Wärmebedarf: {df_full['waermebedarf_MWth'].sum():.0f} MWh")
print(f"  Spitzenlast: {df_full['waermebedarf_MWth'].max():.1f} MW")

---

## Teil 5: Konfiguration für Simulation vorbereiten

Wir müssen die Zeitreihen-Daten mit der Netzwerk-Config verbinden.

In [ ]:
# Lade exportierte Config
with open(export_path, 'r') as f:
    sim_config = yaml.safe_load(f)

# Füge Daten-Pfad hinzu
sim_config['data'] = {
    'input_file': str(Path(data_file).absolute()),
}

# Speichere finale Simulations-Config
sim_config_path = Path('exports/simulation_config.yaml')
with open(sim_config_path, 'w') as f:
    yaml.dump(sim_config, f, default_flow_style=False, sort_keys=False)

print(f"✅ Simulations-Config erstellt: {sim_config_path}")

---

## Teil 6: Simulation ausführen

**WICHTIG:** Dies erfordert Gurobi mit gültiger Lizenz!

In [ ]:
print("▶ Starte Optimierung...")
print("   Dies kann einige Minuten dauern.")
print()

try:
    # Simulation ausführen
    result = run_workflow(
        config_paths=[str(sim_config_path)],
        overrides={
            'data': {'input_file': str(Path(data_file).absolute())}
        }
    )
    
    print("✅ Simulation erfolgreich abgeschlossen!")
    print()
    print("Ergebnis:")
    print(f"  - PF Result: {'✅' if result.pf_result else '❌'}")
    print(f"  - RH Result: {'✅' if result.rh_result else '❌'}")
    print(f"  - MPC Result: {'✅' if result.mpc_result else '❌'}")
    
except Exception as e:
    print(f"❌ Simulation fehlgeschlagen: {e}")
    print()
    print("Häufige Ursachen:")
    print("  1. Gurobi nicht installiert oder Lizenz ungültig")
    print("  2. Zeitreihen-Daten passen nicht zur Config")
    print("  3. Fehlende WRG-Spalten für Wärmepumpen")
    raise

---

## Teil 7: Ergebnisse analysieren

### 7.1 Kosten-Übersicht

In [ ]:
# Primäres Ergebnis (PF, RH oder MPC)
if result.rh_result:
    primary = result.rh_result
    label = "Rolling Horizon"
elif result.mpc_result:
    primary = result.mpc_result
    label = "MPC"
else:
    primary = result.pf_result
    label = "Perfect Forecast"

print(f"Ergebnisse ({label}):")
print("="*60)

# Kosten extrahieren
if hasattr(primary, 'costs') and primary.costs:
    total_cost = 0
    capex = 0
    opex = 0
    
    print("\nKosten-Breakdown:")
    for key, value in primary.costs.items():
        if isinstance(value, (int, float)):
            print(f"  {key:50s}: {value:>15,.2f} EUR")
            total_cost += value
            
            # Kategorisierung
            if 'investment' in key.lower() or 'capex' in key.lower():
                capex += value
            else:
                opex += value
    
    print("\n" + "="*60)
    print(f"  {'CAPEX (Investition)':50s}: {capex:>15,.2f} EUR")
    print(f"  {'OPEX (Betrieb)':50s}: {opex:>15,.2f} EUR")
    print("="*60)
    print(f"  {'GESAMT':50s}: {total_cost:>15,.2f} EUR")
    print("="*60)
else:
    print("⚠️  Keine Kosten-Daten verfügbar")

### 7.2 Optimierte Anlagen-Dimensionierung

In [ ]:
if hasattr(primary, 'summary') and primary.summary:
    print("\nOptimierte Anlagen:")
    print("="*60)
    
    # Heat Pumps
    if 'heat_pumps' in primary.summary:
        print("\nWärmepumpen:")
        for hp_id, hp_data in primary.summary['heat_pumps'].items():
            capacity = hp_data.get('installed_capacity_MW', 'N/A')
            production = hp_data.get('total_heat_production_MWh', 0)
            print(f"  {hp_id:30s}: {capacity:>8} MW | {production:>10,.0f} MWh")
    
    # Storage
    if 'storage' in primary.summary:
        print("\nSpeicher:")
        sto_data = primary.summary['storage']
        capacity = sto_data.get('energy_capacity_MWh', 'N/A')
        power = sto_data.get('power_capacity_MW', 'N/A')
        cycles = sto_data.get('full_cycles', 0)
        print(f"  Kapazität: {capacity} MWh | Leistung: {power} MW | Zyklen: {cycles:.1f}")
    
    # Generators/Boilers
    if 'generators' in primary.summary:
        print("\nKessel/Generatoren:")
        for gen_id, gen_data in primary.summary['generators'].items():
            production = gen_data.get('total_heat_production_MWh', 0)
            print(f"  {gen_id:30s}: {production:>10,.0f} MWh")
    
    print("="*60)
else:
    print("⚠️  Keine Summary-Daten verfügbar")

### 7.3 Zeitreihen visualisieren

In [ ]:
if hasattr(primary, 'table') and primary.table:
    # Zeitreihen-Daten extrahieren
    data = primary.table.data
    
    # Wärmeerzeugung nach Komponente
    heat_cols = [col for col in data.keys() if '_Q_th_MW' in col]
    
    if heat_cols:
        fig, axes = plt.subplots(2, 1, figsize=(14, 8))
        
        # Plot 1: Gestapelte Erzeugung
        df_heat = pd.DataFrame({col: data[col] for col in heat_cols})
        df_heat.plot.area(ax=axes[0], alpha=0.7)
        axes[0].set_ylabel('Wärmeerzeugung (MW)')
        axes[0].set_title('Wärmeerzeugung nach Komponente')
        axes[0].legend(loc='upper left', bbox_to_anchor=(1, 1))
        axes[0].grid(True, alpha=0.3)
        
        # Plot 2: Speicher-SOC
        storage_cols = [col for col in data.keys() if 'SOC' in col and 'TES' in col]
        if storage_cols:
            for col in storage_cols:
                axes[1].plot(data[col], label=col, linewidth=2)
            axes[1].set_ylabel('Füllstand (%)')
            axes[1].set_xlabel('Zeitschritt')
            axes[1].set_title('Speicher-Füllstand (SOC)')
            axes[1].legend()
            axes[1].grid(True, alpha=0.3)
            axes[1].set_ylim([0, 100])
        
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  Keine Wärmeerzeugung-Daten gefunden")
else:
    print("⚠️  Keine Zeitreihen-Daten verfügbar")

### 7.4 Kosten-Visualisierung

In [ ]:
if hasattr(primary, 'costs') and primary.costs:
    # Gruppiere Kosten
    cost_groups = {}
    for key, value in primary.costs.items():
        if isinstance(value, (int, float)) and value > 0:
            # Vereinfachte Kategorien
            if 'investment' in key.lower():
                category = 'Investment (CAPEX)'
            elif 'electricity' in key.lower() or 'grid' in key.lower():
                category = 'Strom'
            elif 'fuel' in key.lower() or 'gas' in key.lower():
                category = 'Brennstoff'
            elif 'co2' in key.lower():
                category = 'CO2'
            else:
                category = 'Sonstige'
            
            cost_groups[category] = cost_groups.get(category, 0) + value
    
    # Pie Chart
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']
    wedges, texts, autotexts = ax.pie(
        cost_groups.values(),
        labels=cost_groups.keys(),
        autopct='%1.1f%%',
        colors=colors,
        startangle=90
    )
    
    # Verbessere Lesbarkeit
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(12)
        autotext.set_weight('bold')
    
    ax.set_title('Kosten-Verteilung', fontsize=14, fontweight='bold')
    
    # Legende mit Werten
    legend_labels = [f"{cat}: {val:,.0f} EUR" for cat, val in cost_groups.items()]
    ax.legend(legend_labels, loc='upper left', bbox_to_anchor=(1, 1))
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Keine Kosten-Daten für Visualisierung")

---

## Teil 8: Interactive Dashboard (optional)

Für detailliertere Analyse das vollständige Dashboard nutzen:

In [ ]:
# Erstelle Dashboard (optional - öffnet in neuem Fenster)
# dashboard = create_dashboard(result, title="Brownfield Optimization Results")
# dashboard.show()

print("💡 Tipp: Für interaktives Dashboard:")
print("   python start_dashboard.py --dir saved_workflows/")

---

## Teil 9: Ergebnisse exportieren

In [ ]:
# Export-Verzeichnis
export_dir = Path('exports/results')
export_dir.mkdir(parents=True, exist_ok=True)

# 1. Kosten als CSV
if hasattr(primary, 'costs') and primary.costs:
    costs_df = pd.DataFrame([
        {'Category': k, 'Value_EUR': v} 
        for k, v in primary.costs.items() 
        if isinstance(v, (int, float))
    ])
    costs_df.to_csv(export_dir / 'costs.csv', index=False)
    print(f"✅ Kosten exportiert: {export_dir / 'costs.csv'}")

# 2. Zeitreihen als CSV
if hasattr(primary, 'table') and primary.table:
    df_timeseries = pd.DataFrame(primary.table.data)
    df_timeseries.to_csv(export_dir / 'timeseries.csv', index=False)
    print(f"✅ Zeitreihen exportiert: {export_dir / 'timeseries.csv'}")

# 3. Summary als YAML
if hasattr(primary, 'summary') and primary.summary:
    with open(export_dir / 'summary.yaml', 'w') as f:
        yaml.dump(primary.summary, f, default_flow_style=False)
    print(f"✅ Summary exportiert: {export_dir / 'summary.yaml'}")

print(f"\n✅ Alle Ergebnisse exportiert nach: {export_dir}")

---

## Zusammenfassung

### Was haben wir gemacht?

1. ✅ **Netzwerk erstellt** (Brownfield-Szenario mit 4 Komponenten)
2. ✅ **Validiert** (Topologie und Eigenschaften geprüft)
3. ✅ **Exportiert** (YAML-Konfiguration generiert)
4. ✅ **Zeitreihen** (Daten vorbereitet oder generiert)
5. ✅ **Simuliert** (Pyomo/Gurobi Optimierung durchgeführt)
6. ✅ **Analysiert** (Kosten, Dimensionierung, Zeitreihen)
7. ✅ **Exportiert** (Ergebnisse als CSV/YAML gespeichert)

### Nächste Schritte

**Iterativer Workflow:**
1. Im Network Designer Komponenten anpassen
2. Neu exportieren
3. Simulation wiederholen
4. Ergebnisse vergleichen

**Erweiterte Analysen:**
- Sensitivitätsanalysen (Strompreis, CO2-Preis)
- Rolling Horizon statt Perfect Forecast
- MPC mit Prognoseunsicherheiten
- Längere Zeiträume (ganzes Jahr)

**Dashboard nutzen:**
```bash
python start_network_designer.py    # Netzwerk-Design
python start_dashboard.py           # Ergebnis-Analyse
```

---

**🎉 Workflow abgeschlossen!**